# Nexto Distillation — Diagnostics + DAgger

This notebook runs deep diagnostics on the BC-trained student, then
optionally runs a DAgger iteration to improve online performance.

**Runtime:** Go to **Runtime → Change runtime type → T4 GPU** before starting.

## 1. Setup

In [ ]:
!git clone https://github.com/AdamTheGans/Rocket-League-Bot.git
%cd Rocket-League-Bot

# Install deps
!pip install -q rlgym[rl-sim]
!pip install -q git+https://github.com/AechPro/rlgym-ppo
!pip install -q numpy==1.26.4 cloudpickle==3.1.2

# Verify
import torch, RocketSim, rlgym, os
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
assert os.path.isfile('nexto/nexto-model.pt'), 'Nexto model not found!'
print('All checks passed.')

## 2. Upload Student Checkpoint

Upload your `student_policy.pt` and `metadata.json` files.

In [ ]:
import os
os.makedirs('checkpoints/nexto_distill', exist_ok=True)

from google.colab import files
print('Upload student_policy.pt:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'checkpoints/nexto_distill/{name}', 'wb') as f:
        f.write(data)

print('\nUpload metadata.json:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'checkpoints/nexto_distill/{name}', 'wb') as f:
        f.write(data)

print('\nFiles in checkpoints/nexto_distill/:')
!ls -la checkpoints/nexto_distill/

## 3. Generate Fresh Dataset for Diagnostics

We need shards for Check 4 (confusion analysis). Generating 200K steps is fast enough.

In [ ]:
%cd src

!python -m nexto_distill.generate_dataset \
    --out_dir ../data/nexto_distill/shards \
    --num_steps 200000 \
    --shard_size 50000 \
    --seed 42 \
    --device cpu \
    --report_every 50000

## 4. Run Full Diagnostics

This runs all 5 checks:
1. **Teacher Baseline** — how does the teacher perform? (50 episodes)
2. **On-Policy Agreement** — student accuracy on teacher-driven states (100K steps)
3. **Student Action Distribution** — action diversity during student rollout (50 episodes)
4. **Confusion Hotspots** — per-action accuracy on val set
5. **Recommendation** — automated go/no-go

In [ ]:
!python -m nexto_distill.diagnostics \
    --checkpoint ../checkpoints/nexto_distill/student_policy.pt \
    --data_dir ../data/nexto_distill/shards \
    --episodes 50 \
    --on_policy_steps 100000

## 5. DAgger Collection

The **student drives**, but each state is labeled by the **teacher**.
This produces on-policy training data to fix covariate shift.

Collect 1M steps (same as original dataset).

In [ ]:
!python -m nexto_distill.dagger_collect \
    --checkpoint ../checkpoints/nexto_distill/student_policy.pt \
    --out_dir ../data/nexto_distill/dagger_shards \
    --num_steps 1000000 \
    --shard_size 50000 \
    --seed 42 \
    --report_every 10000

## 6. Retrain with Original + DAgger Data

Train on the original teacher-driven shards + DAgger on-policy shards combined.

In [ ]:
!python -m nexto_distill.train_bc \
    --data_dir ../data/nexto_distill/shards \
    --extra_data_dirs ../data/nexto_distill/dagger_shards \
    --layers 2048 1024 1024 512 \
    --lr 3e-4 \
    --epochs 30 \
    --batch_size 4096 \
    --checkpoint_dir ../checkpoints/nexto_distill_dagger \
    --device cuda \
    --seed 42

## 7. Re-Evaluate After DAgger

In [ ]:
# Offline eval
!python -m nexto_distill.eval_imitation \
    --checkpoint ../checkpoints/nexto_distill_dagger/student_policy.pt \
    --mode offline \
    --data_dir ../data/nexto_distill/shards \
    --device cuda

print('\n' + '='*60 + '\n')

# Online eval
!python -m nexto_distill.eval_imitation \
    --checkpoint ../checkpoints/nexto_distill_dagger/student_policy.pt \
    --mode online \
    --episodes 50 \
    --device cpu

## 8. Download Results

In [ ]:
!zip -r /content/nexto_distill_dagger_results.zip \
    ../checkpoints/nexto_distill_dagger/ \
    ../data/nexto_distill/shards/metadata.json \
    ../data/nexto_distill/dagger_shards/metadata.json

from google.colab import files
files.download('/content/nexto_distill_dagger_results.zip')